# Code for formatting ANVIL-generated episode annotations

## Imports

In [1]:
from pathlib import Path
from typing import Union

import numpy as np
import pandas as pd

from analysis_helpers.constants import ANNOTATIONS_DIR

## Set up params for replacing column names & null values

In [2]:
dtypes = {
    'Frame': 'Int64', 
    'Time': 'Float64', 
    'Scene info:narrative details - external': 'string', 
    'Scene info:narrative details - internal': 'string', 
    'Scene info:characters on screen': 'string', 
    'Scene info:Music Presence': 'string', 
    'speech:transcription': 'string', 
    'speech:character speaking': 'string', 
    'setting:indoor/outdoor': 'Int64', 
    'setting:setting': 'string', 
    'Scene name:scene name': 'string'
}
old_cols = dtypes.keys()
new_cols = (
    'Frame', 
    'Onset time', 
    'Narrative details (external events)', 
    'Narrative details (internal state)', 
    'Characters on screen', 
    'Music presence', 
    'Speech', 
    'Character speaking', 
    'Indoor/outdoor', 
    'Setting', 
    'Scene name'
)
replace_values = {
    'Narrative details (external events)': {'0': pd.NA},
    'Narrative details (internal state)': {'0' : pd.NA},
    'Characters on screen': {'0': pd.NA},
    'Music presence': {'0': 'no', '1': 'yes'},
    'Speech': {'0': pd.NA, '-1000': pd.NA},
    'Character speaking': {'0': pd.NA, '-1000': pd.NA},
    'Indoor/outdoor': {1: 'indoor', 2: 'outdoor', 0: pd.NA, -1000: pd.NA},
    'Setting': {'0': pd.NA, '-1000': pd.NA},
    'Scene name': {'-1000': pd.NA}
}

### functions

In [3]:
def format_annotations(raw_path: Union[Path, str]) -> pd.DataFrame:
    """
    formats ANVIL-generated annotations into a pandas.DataFrames
    """
    df = pd.read_csv(raw_path, sep='\t', usecols=old_cols, dtype=dtypes)
    df.columns = new_cols
    # drop consecutive duplicate annotations, keep first frame only 
    # (also account for occasional 2-6 frame delay between start of 
    # Narrative details and Speech annotation blocks)
    nd_external = 'Narrative details (external events)'
    df = df.loc[
        (df[nd_external].shift(fill_value='') != df[nd_external]) | 
        (df['Speech'].shift(fill_value='') != df['Speech'])
    ]
    df = df.loc[df[nd_external].shift(-1, fill_value='') != df[nd_external]]
    df.replace(replace_values, inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

### load annotations, format, and save out

In [4]:
for episode in ('atlep1', 'atlep2', 'arrdev'):
    raw_path = ANNOTATIONS_DIR.joinpath(f'{episode}-raw.tsv')
    annot_df = format_annotations(raw_path)
#     annot_df.to_csv(raw_path.with_name(f'{episode}.csv'), index=False)